# Experanto experiment with Allen dataset

This notebook downloads and exports allen dataset (using allen-exporter) and shows how to use experanto to interpolate the experiment generally and its individual modalities.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdelrahman725/experanto/blob/allen-dataset-colab-example/examples/allen_example.ipynb)


# 1. Export allen dataset using allen-exporter

## Important (for developers)

Default Python version in colab is "Python 3.12", but `allensdk` (which allen-exporter uses to download the experiment data) does NOT work with that version. While you can simply change Runtime version to "2025.07" which supports Python 3.11.13, that "2025.07" version will no longer be supported on "July 2026" (after a year of its release). So for consistency, we stick to set up our environment using **Micromamba** (a minimal package and environment manager) which gives us Python 3.11 environment.

All shell commands (only in step 1.) should be run using this command: `!micromamba run -n py311 command_name`


See [colab supported Runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html#)

## Setup Micromamba environment

In [ ]:
# Install Micromamba (fast, minimal implementation of Conda.)
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
!mv bin/micromamba /usr/local/bin/

# Verify
!micromamba --version

# Create Conda environment with Python 3.11
!micromamba create -y -n py311 python=3.11

# Verify output is 3.11
!micromamba run -n py311 python --version

In [ ]:
# Install allen-exporter inside Micromamba environment.
!micromamba run -n py311 pip install git+https://github.com/sensorium-competition/allen-exporter.git "setuptools<70" matplotlib-inline

## Download and export Allen dataset

In [ ]:
# Run cell inside Micromamba
%%bash
micromamba run -n py311 python << 'PY'

import allensdk
import warnings
from allen_exporter.exporter import multi_session_export

# Verify allensdk is installed.
print("Allensdk version : ", allensdk.__version__)
print("\nImport confirmed\n")

# Suppress warnings that are not important.
warnings.filterwarnings("ignore")

# Download (if it's first time) and export Allen dataset, this takes ~1 min.
# After export, all data should be available at /data.
cache, ids = multi_session_export(1, val_rate= 0.2, subsample_frac=1)

PY

# 2. Experanto

## Setup environment for experanto

**Note:** After running the next cell, you may be prompted to "Restart session". Don't worry, the experiment `data/` folder (created in the previous step) persists across sessions.

In [ ]:
# Install experanto in default colab environment
!pip install experanto git+https://github.com/sensorium-competition/experanto.git

## Load experiment

In [1]:
import os
from experanto.experiment import Experiment

parent_dir = './data/allen_data'
try:
  # Compute path to the experiment dir dynamically.
  matches = [
      d for d in os.listdir(parent_dir) if d.startswith('experiment')
      and os.path.isdir(os.path.join(parent_dir, d))
  ]
  if len(matches) == 0:
    raise FileNotFoundError()

except FileNotFoundError:
    raise FileNotFoundError("No experiment directory found. Make sure you run step 1 correctly")

experiment_dir = os.path.join(parent_dir, matches[0])

# Load the exported experiment.
exp = Experiment(experiment_dir)

# See compatible modalities for the experiment
print("Available experiment devices:", exp.devices.keys())

Available experiment devices: dict_keys(['screen', 'treadmill', 'eye_tracker', 'responses'])


/usr/local/lib/python3.12/dist-packages/experanto/experiment.py:113: UserWarning: Falling back to original Interpolator creation logic.
  warnings.warn(


## Interpolation

### Interpolate from all devices

In [3]:
import numpy as np

# Query 100 time points spread evenly over 10 seconds
times = np.linspace(0, 10, 100)

# Get interpolated data from all devices
data = exp.interpolate(times)

for device, device_data in data.items():
    print(device, ":\n")
    print(device_data)
    print("\n--------------------\n")

screen :

[[[127.99999 128.      127.99999 ... 128.      128.      127.99999]
  [127.99999 128.      127.99999 ... 128.      128.      127.99999]
  [127.99999 128.      127.99999 ... 128.      128.      127.99999]
  ...
  [127.99999 128.      127.99999 ... 128.      128.      127.99999]
  [127.99999 128.      127.99999 ... 128.      128.      127.99999]
  [127.99999 128.      127.99999 ... 128.      128.      127.99999]]

 [[127.99999 128.      127.99999 ... 128.      128.      127.99999]
  [127.99999 128.      127.99999 ... 128.      128.      127.99999]
  [127.99999 128.      127.99999 ... 128.      128.      127.99999]
  ...
  [127.99999 128.      127.99999 ... 128.      128.      127.99999]
  [127.99999 128.      127.99999 ... 128.      128.      127.99999]
  [127.99999 128.      127.99999 ... 128.      128.      127.99999]]

 [[127.99999 128.      127.99999 ... 128.      128.      127.99999]
  [127.99999 128.      127.99999 ... 128.      128.      127.99999]
  [127.99999 128.     

### Interpolate from a specific device

In [4]:
import numpy as np

# Query 100 time points spread evenly over 10 seconds
times = np.linspace(0, 10, 100)


# device can be: screen, treadmill, eye_tracker, or responses
screen = exp.interpolate(times, device="screen")
treadmill = exp.interpolate(times, device="treadmill")
eye_tracker = exp.interpolate(times, device="eye_tracker")
responses = exp.interpolate(times, device="responses")

# Change here what device you want to see its interpolated signals
print(responses)

[[ 0.93657291  0.55576146  0.25939912  0.41124767  0.08977238  0.19020046
   0.57376695  2.90015626  0.4239428   0.62480003  0.29607722  0.28595215]
 [ 0.58248562  0.74787319  0.30149347  0.19771612  0.07466755  0.02105861
   0.41276214  0.96703732  0.60503817  0.13580583  0.08253972  0.18900922]
 [ 1.29600501  0.43970796  0.34024861  0.14388797  0.26002109  0.15665245
   0.12951383  0.56966907  0.64048272  0.10336801  0.23056522  0.01171901]
 [ 0.84489775 -0.04886976  0.12116835  0.13635454  0.34542489  0.13520737
   0.34949198  0.08619506  0.20034993  0.24432643  0.04468284  0.27774701]
 [ 1.18118751  0.2559959   0.05955438  0.02371915  0.22732048  0.24450895
   0.53216243 -0.83702052  0.65515441  0.02353403  0.07673727  0.24609959]
 [ 1.27471638  0.10850338  0.18655133  0.06242622  0.14956538  0.00996919
   0.09380771  0.16637991 -0.16068403  0.06777227  0.43875194  0.23804431]
 [ 1.39416265  0.10290431 -0.02241463  0.07999286  0.2551544   0.13212764
   0.2290148   0.86402273  0.080